# Khmer Handwritten Word Recognition — Colab Training Runner

Runs all three approaches (A1 baseline, A2 transfer learning, A3 Transformer) on a Colab GPU, with **all data, checkpoints, and results stored on Google Drive** — nothing large touches this session's local disk except the cloned code, and nothing persists locally on your Mac at all.

**Before running:** upload your dataset to Google Drive at
`MyDrive/khmer-ocr/data/metadata/labels.csv` (columns: `image,label,writer_id`) and the corresponding images under `MyDrive/khmer-ocr/data/raw/`.

First run creates the tokenizer vocabulary and the writer-disjoint split and saves both to Drive; every later run (any approach, any session) automatically reuses them — this is what keeps the 3-approach comparison fair, per the course rubric's "same test set for every approach" rule.

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = '/content/drive/MyDrive/khmer-ocr'
import os
for sub in ['data/metadata', 'data/raw', 'checkpoints', 'results']:
    os.makedirs(f'{DRIVE_ROOT}/{sub}', exist_ok=True)
print('Drive folders ready under', DRIVE_ROOT)

## 2. Get the project code
Code is cloned fresh into this session's fast local disk (`/content/`) — only data/checkpoints/results live on Drive. If the repo is private, use a GitHub personal access token in the URL instead.

In [ ]:
%cd /content
!rm -rf khmer-handwritten-word-recognition
!git clone https://github.com/Thna17/khmer-handwritten-word-recognition.git
%cd khmer-handwritten-word-recognition
!git log --oneline -5

## 3. Install dependencies
(Colab already ships torch/torchvision with GPU support — this only adds what's missing, e.g. `jiwer`.)

In [ ]:
!pip install -q -r requirements.txt
import torch
print('torch', torch.__version__, '| CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
else:
    print('WARNING: no GPU detected — in the Colab menu, go to Runtime > Change runtime type > GPU')

## 4. Point every path at Google Drive
Edit `METADATA_CSV` / `IMAGES_DIR` if your filenames differ.

In [ ]:
METADATA_CSV   = f'{DRIVE_ROOT}/data/metadata/labels.csv'
IMAGES_DIR     = f'{DRIVE_ROOT}/data/raw'
CHECKPOINT_DIR = f'{DRIVE_ROOT}/checkpoints'
RESULTS_DIR    = f'{DRIVE_ROOT}/results'
VOCAB_PATH     = f'{DRIVE_ROOT}/data/metadata/char_to_idx.json'
SPLIT_PATH     = f'{DRIVE_ROOT}/data/metadata/writer_split.json'

assert os.path.exists(METADATA_CSV), f'Upload your metadata CSV to {METADATA_CSV} first'
import csv
with open(METADATA_CSV, encoding='utf-8') as f:
    n_rows = sum(1 for _ in csv.DictReader(f))
print(f'{n_rows} labeled samples found at {METADATA_CSV}')

## 5. Sanity check: run the fast test suite before spending GPU time
(Skips the ~4-minute tiny-overfit gate test — that one's for local dev, not every Colab run.)

In [ ]:
!python -m pytest tests/ -q --ignore=tests/test_overfit.py

## 6. Train A1 — Baseline CRNN (VGG CNN from scratch + BiLSTM)
First invocation of any approach creates `char_to_idx.json` and `writer_split.json` on Drive; every cell below reuses them automatically.

In [ ]:
!python -m src.run_experiment \
  --approach baseline \
  --metadata-csv "{METADATA_CSV}" --images-dir "{IMAGES_DIR}" \
  --checkpoint-dir "{CHECKPOINT_DIR}" --results-dir "{RESULTS_DIR}" \
  --vocab-path "{VOCAB_PATH}" --split-path "{SPLIT_PATH}" \
  --batch-size 32 --max-epochs 100 --patience 10

## 7. Train A2 — Transfer Learning (ResNet-18 backbone)
Run twice: frozen backbone (linear probe) and fully fine-tuned — this pairing is itself the required E010/E011 experiment, see `EXPERIMENTS.md`.

In [ ]:
!python -m src.run_experiment \
  --approach transfer --freeze-backbone --checkpoint-prefix a2_transfer_frozen \
  --metadata-csv "{METADATA_CSV}" --images-dir "{IMAGES_DIR}" \
  --checkpoint-dir "{CHECKPOINT_DIR}" --results-dir "{RESULTS_DIR}" \
  --vocab-path "{VOCAB_PATH}" --split-path "{SPLIT_PATH}" \
  --batch-size 32 --max-epochs 100 --patience 10

In [ ]:
!python -m src.run_experiment \
  --approach transfer --checkpoint-prefix a2_transfer_finetuned \
  --metadata-csv "{METADATA_CSV}" --images-dir "{IMAGES_DIR}" \
  --checkpoint-dir "{CHECKPOINT_DIR}" --results-dir "{RESULTS_DIR}" \
  --vocab-path "{VOCAB_PATH}" --split-path "{SPLIT_PATH}" \
  --batch-size 32 --max-epochs 100 --patience 10

## 8. Train A3 — Transformer sequence encoder (same CNN as A1)

In [ ]:
!python -m src.run_experiment \
  --approach transformer \
  --metadata-csv "{METADATA_CSV}" --images-dir "{IMAGES_DIR}" \
  --checkpoint-dir "{CHECKPOINT_DIR}" --results-dir "{RESULTS_DIR}" \
  --vocab-path "{VOCAB_PATH}" --split-path "{SPLIT_PATH}" \
  --batch-size 32 --max-epochs 100 --patience 10

## 9. Compare all approaches on the identical test set

In [ ]:
import json, glob
import pandas as pd

rows = []
for path in sorted(glob.glob(f'{RESULTS_DIR}/*_results.json')):
    with open(path, encoding='utf-8') as f:
        r = json.load(f)
    m = r['test_metrics']
    rows.append({
        'checkpoint': path.split('/')[-1].replace('_results.json', ''),
        'approach': r['approach'],
        'params': r['num_trainable_params'],
        'CER %': round(m['cer'] * 100, 2),
        'WER %': round(m['wer'] * 100, 2),
        'WordAcc %': round(m['word_accuracy'] * 100, 2),
        'epochs_run': len(r['history']),
    })

comparison = pd.DataFrame(rows).sort_values('CER %')
comparison

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].bar(comparison['checkpoint'], comparison['CER %'])
axes[0].set_title('Test CER (%) — lower is better')
axes[0].tick_params(axis='x', rotation=30)
axes[1].bar(comparison['checkpoint'], comparison['WordAcc %'])
axes[1].set_title('Test Word Accuracy (%) — higher is better')
axes[1].tick_params(axis='x', rotation=30)
fig.tight_layout()
fig.savefig(f'{RESULTS_DIR}/comparison_chart.png', dpi=120)
plt.show()
print(f'Saved to {RESULTS_DIR}/comparison_chart.png')